In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import linear_kernel
from ast import literal_eval
import numpy as np

In [2]:
import pandas as pd
df = pd.read_csv('./data/movies_metadata.csv', low_memory=False)
df.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               45466 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

# Fórmula de Media Ponderada

---

$$
\text{media ponderada} = \left( \frac{v}{v + m} \times R \right) + \left( \frac{m}{v + m} \times C \right)
$$

## Variables:

- **v** = número de votos de la película
- **m** = número mínimo de votos requeridos para ser incluido en la informe
- **R** = calificación promedio de la película
- **C** = media de los votos en todo el informe

---

In [4]:
# Como primer paso, calculemos el valor de C, la calificación media de todas las películas
C = df['vote_average'].mean()
print(f"C= {C}")

C= 5.618207215134185


In [5]:
m = df['vote_count'].quantile(0.90)
print(f"m= {m}")

m= 160.0


In [6]:
# peliculas que han recibido un percentil de votos mayor o igual a m (90)
# Con esto vemos que hay alrededor del 10% de las películas con más de 160 votos, las cuales califican para estar en esta lista.

q_movies = df.copy().loc[df['vote_count'] >= m]
print(f"shape de q_movies: {q_movies.shape} , shape de df: {df.shape}")

shape de q_movies: (4555, 24) , shape de df: (45466, 24)


In [8]:
# media ponderada
def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']

    # Cálculo de IMDB
    return (v/(v+m) * R) + (m/(m+v) * C)

In [9]:
q_movies['score'] = q_movies.apply(weighted_rating, axis=1)

In [10]:
q_movies = q_movies.sort_values('score', ascending=False)

#Mostrar los primeros 15 resultados
q_movies[['title', 'vote_count', 'vote_average', 'score']].head(15)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


---
# Recomendadores basados en contenido


Para calcular la realizar la recomendacion basada en contenido revisamos la columna "overview" vemos que ahora requerimos PNL

In [10]:
df['overview'].head()

0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
Name: overview, dtype: object

#### Vectorización TF-IDF

In [21]:
#Por lo tanto, cada película será un vector columna de 1x45466, donde cada columna representará una puntuación de similitud con cada película.
# Calcular la matriz de similitud coseno
#
#  NOTA
#
# Despues de muchos intentos no pude ejecutar cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
# debido a problemas de memoria que reiniciaban constantemente el server de jupyter-lab
# una alternativa es acotarlo usando q_movies (otra alternativa es un analisis completo del dataset original pero esto me parecio mas simple)

df_popular = df[df['vote_count'] > 5000].copy()
print(f"Peliculas populares: {df_popular.shape[0]}")

# reset_index para alinear indices con la matriz TF-IDF
df_popular = df_popular.reset_index(drop=True)

# TF-IDF reducido sobre populares
tfidf_reducido = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    min_df=2,
    max_df=0.8
)
df_popular['overview'] = df_popular['overview'].fillna('')
tfidf_matrix_reducido = tfidf_reducido.fit_transform(df_popular['overview'])

print(f"Shape reducido: {tfidf_matrix_reducido.shape}")
n = tfidf_matrix_reducido.shape[0]
print(f"Memoria estimada para cosine_sim: {n**2 * 8 / 1e9:.2f} GB")


Peliculas populares: 101
Shape reducido: (101, 421)
Memoria estimada para cosine_sim: 0.00 GB


#### Tenemos 421 palabras diferentes basado en los mas populares.

In [22]:
tfidf.get_feature_names_out()[5000:5010]

array(['avails', 'avaks', 'avalanche', 'avalanches', 'avallone', 'avalon',
       'avant', 'avanthika', 'avanti', 'avaracious'], dtype=object)

In [23]:

# Calcular similitud solo si es manejable
if n < 15000:
    cosine_sim = linear_kernel(tfidf_matrix_reducido, tfidf_matrix_reducido)
    print(f"Matriz creada: {cosine_sim.shape}")
else:
    print(f"Aun grande ({n} peliculas), reducir umbral de votos a 10000+")







Matriz creada: (101, 101)


In [26]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
indices[:10]

title
Toy Story                      0
Jumanji                        1
Grumpier Old Men               2
Waiting to Exhale              3
Father of the Bride Part II    4
Heat                           5
Sabrina                        6
Tom and Huck                   7
Sudden Death                   8
GoldenEye                      9
dtype: int64

In [29]:
# Construir el mapa de indices UNA SOLA VEZ (despues del reset_index)
# df_popular ya debe tener reset_index(drop=True) aplicado
indices = pd.Series(df_popular.index, index=df_popular['title'].str.lower()).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim, top_n=10):
    """
    Recomienda peliculas similares basadas en contenido (overview TF-IDF).
    
    Parametros:
        title   : str  - Titulo de la pelicula (no sensible a mayusculas)
        cosine_sim: np.ndarray - Matriz de similitud precalculada
        top_n   : int  - Numero de recomendaciones a devolver
    
    Retorna:
        DataFrame con title, vote_average, vote_count, overview
    """
    title_lower = title.lower()
    
    # Validar que la pelicula existe
    if title_lower not in indices:
        print(f"Pelicula '{title}' no encontrada en peliculas populares.")
        print("Sugerencias:")
        sugerencias = [t for t in indices.index if title_lower[:4] in t][:5]
        for s in sugerencias:
            print(f"  - {s}")
        return None
    
    # Obtener indice alineado con la matriz TF-IDF
    idx = indices[title_lower]
    
    # Calcular puntuaciones de similitud
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Ordenar por similitud descendente
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Excluir la propia pelicula (posicion 0) y tomar top_n
    sim_scores = sim_scores[1:top_n + 1]
    
    # Extraer indices
    movie_indices = [i[0] for i in sim_scores]
    
    # Retornar con informacion enriquecida (no solo titulo)
    resultado = df_popular.iloc[movie_indices][['title', 'vote_average', 'vote_count', 'overview']].copy()
    resultado['similarity_score'] = [round(i[1], 4) for i in sim_scores]
    
    return resultado


In [30]:
get_recommendations('The Dark Knight Rises')

,title,vote_average,vote_count,overview,similarity_score
32,The Dark Knight,8.3,12269.0,Batman raises the stakes in his war on crime. ...,0.4276
27,Batman Begins,7.5,7511.0,"Driven by tragedy, billionaire Bruce Wayne ded...",0.2672
94,Batman v Superman: Dawn of Justice,5.7,7189.0,Fearing the actions of a god-like Super Hero l...,0.2126
43,Despicable Me,7.1,6595.0,Villainous Gru lives up to his reputation as a...,0.1032
91,Inside Out,7.9,6737.0,"Growing up can be a bumpy road, and it's no ex...",0.0979
86,Ant-Man,7.0,6029.0,Armed with the astonishing ability to shrink i...,0.0964
17,The Lord of the Rings: The Fellowship of the Ring,8.0,8892.0,"Young hobbit Frodo Baggins, after inheriting a...",0.0912
75,Guardians of the Galaxy,7.9,10014.0,"Light years from Earth, 26 years after being a...",0.0836
1,Se7en,8.1,5915.0,Two homicide detectives are on a desperate hun...,0.0755
10,Titanic,7.5,7770.0,"84 years later, a 101-year-old woman named Ros...",0.0728


In [31]:
get_recommendations('The Godfather')

,title,vote_average,vote_count,overview,similarity_score
3,Pulp Fiction,8.3,8670.0,"A burger-loving hit man, his philosophical par...",0.2286
65,World War Z,6.7,5683.0,Life for former United Nations investigator Ge...,0.1933
14,Gladiator,7.9,5566.0,"In the year 180, the death of emperor Marcus A...",0.1484
68,Frozen,7.3,5440.0,Young princess Anna of Arendelle dreams about ...,0.1374
75,Guardians of the Galaxy,7.9,10014.0,"Light years from Earth, 26 years after being a...",0.1192
32,The Dark Knight,8.3,12269.0,Batman raises the stakes in his war on crime. ...,0.1009
36,Up,7.8,7048.0,Carl Fredricksen spent his entire life dreamin...,0.0946
58,Life of Pi,7.2,5912.0,"The story of an Indian boy named Pi, a zookeep...",0.0945
43,Despicable Me,7.1,6595.0,Villainous Gru lives up to his reputation as a...,0.0823
37,The Hangover,7.2,6324.0,When three friends finally come to after a rau...,0.0768


In [3]:
# cargar conjuntos de datos adicionales
credits = pd.read_csv('./data/credits.csv')
keywords = pd.read_csv('./data/keywords.csv')

credits.info()
keywords.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45476 entries, 0 to 45475
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   cast    45476 non-null  object
 1   crew    45476 non-null  object
 2   id      45476 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.0+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46419 entries, 0 to 46418
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        46419 non-null  int64 
 1   keywords  46419 non-null  object
dtypes: int64(1), object(1)
memory usage: 725.4+ KB


In [4]:

# Eliminar algunos IDs problemáticos
df = df.drop([19730, 29503, 35587, 35803])

# Convetir todos los ids a números enteros
keywords['id'] = keywords['id'].astype('int')
credits['id'] = credits['id'].astype('int')
df['id'] = df['id'].astype('int')

# Hacer merges entre data frames
df = df.merge(credits, on='id')
df = df.merge(keywords, on='id')

In [5]:

features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(literal_eval)

In [6]:
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [7]:
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        #Checar si existen más de 3 elementos. Si sí, regresar primeros 3, si no, todos
        if len(names) > 3:
            names = names[:3]
        return names

    # regresar lista vacía si los datos no están bien formateados
    return []

In [8]:
# Extraer director de columnas crew
df['director'] = df['crew'].apply(get_director)

# Extraer top 3 de elenco, palabras clave y géneros
features = ['cast', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(get_list)

In [9]:
df[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

,title,cast,director,keywords,genres
0,Toy Story,"[Tom Hanks, Tim Allen, Don Rickles]",John Lasseter,"[jealousy, toy, boy]","[Animation, Comedy, Family]"
1,Jumanji,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]",Joe Johnston,"[board game, disappearance, based on children'...","[Adventure, Fantasy, Family]"
2,Grumpier Old Men,"[Walter Matthau, Jack Lemmon, Ann-Margret]",Howard Deutch,"[fishing, best friend, duringcreditsstinger]","[Romance, Comedy]"


In [10]:
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        #Checar si existse el director. Si no, regresar ""
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''

In [11]:
features = ['cast', 'keywords', 'director', 'genres']

for feature in features:
    df[feature] = df[feature].apply(clean_data)

In [12]:
def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])

df['soup'] = df.apply(create_soup, axis=1)

df[['soup']].head(2)

,soup
0,jealousy toy boy tomhanks timallen donrickles ...
1,boardgame disappearance basedonchildren'sbook ...


In [13]:

count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df['soup'])


count_matrix.shape

(46626, 73880)

In [ ]:

cosine_sim2 = cosine_similarity(count_matrix, count_matrix)

# Reset index of your main DataFrame and construct reverse mapping as before
df = df.reset_index()
indices = pd.Series(df.index, index=df['title'])